In [ ]:
# computing compound scores with VADER
# https://vadersentiment.readthedocs.io/en/latest/

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

df = pd.read_csv('MASTER_SAMPLED.csv') 

analyzer = SentimentIntensityAnalyzer()

def get_compound_score(text):
    if pd.isna(text): return 0          # check if there is text
    return analyzer.polarity_scores(str(text))['compound']    # compute polarity scores per text

df['sentiment_score'] = df['full_text'].apply(get_compound_score) # apply column to dataset

regional_summary = df.groupby('region')['sentiment_score'].agg(['mean', 'std', 'count']).reset_index() # mean & s.d.

def categorize(score):          # labelling
    if score >= 0.05: return "positive"
    elif score <= -0.05: return "negative"
    else: return "neutral"

regional_summary['label'] = regional_summary['mean'].apply(categorize)

regional_summary.columns = ['region', 'avg_sentiment', 's.d.', 'article_count', 'overall_tone']
regional_summary = regional_summary.round(4)

regional_summary.to_csv('vadersentiment.csv', index=False)

print(regional_summary)

In [ ]:
# label distribution and score distribution plots for VADER , kruskal-wallis and mann-whitney U tests
# https://gist.github.com/lundquist-ecology-lab/48660ed53969d8ddacd308bd71ed7f65
 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np
from scipy import stats
 
df = pd.read_csv("MASTER_SAMPLED.csv")
df["date_parsed"] = pd.to_datetime(
    df["date"], format="%a, %d %b %Y %H:%M:%S %Z", utc=True
)
df = df.sort_values("date_parsed").reset_index(drop=True)
 
analyzer = SentimentIntensityAnalyzer() # score for every article
 
def get_vader_scores(text):
    if pd.isna(text):
        return {"compound": 0, "pos": 0, "neu": 0, "neg": 0}
    return analyzer.polarity_scores(str(text))
 
scores = df["full_text"].apply(get_vader_scores)
df["vader_compound"] = scores.apply(lambda x: x["compound"])
df["vader_pos"]      = scores.apply(lambda x: x["pos"])
df["vader_neu"]      = scores.apply(lambda x: x["neu"])
df["vader_neg"]      = scores.apply(lambda x: x["neg"])
 
def categorize(score):
    if score >= 0.05:  return "positive"
    if score <= -0.05: return "negative"
    return "neutral"
 
df["vader_label"] = df["vader_compound"].apply(categorize)
df["month"]       = df["date_parsed"].dt.to_period("M")
df["week"]        = df["date_parsed"].dt.to_period("W")
 
df.to_csv("vadersentiment.csv", index=False)
 
reg_order  = ["US", "affected europe", "greenland & denmark", "other"]
reg_colors = {'US': 'indianred', 
           'affected europe': 'darkgoldenrod', 
           'greenland & denmark': 'darkseagreen', 
           'other': 'lightslategrey'}
palette = [reg_colors[r] for r in reg_order]
 
# summary table
summary = (
    df.groupby("region")["vader_compound"]
    .agg(["mean", "std", "count"])
    .reindex(reg_order)
    .rename(columns={"mean": "avg_compound", "std": "sd", "count": "n"})
    .round(4)
)
label_pct = (
    df.groupby("region")["vader_label"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .unstack(fill_value=0)
    .reindex(reg_order)
)
summary = summary.join(label_pct)
print("\nSUMMARY TABLE:")
print(summary.to_string())
 
# stat tests (kruskal)
groups = [
    df[df["region"] == r]["vader_compound"].values
    for r in reg_order
]
H_stat, p_kruskal = stats.kruskal(*groups)
print(f"\nkruskal-wallis H={H_stat:.3f}, p={p_kruskal:.4f}")
print("regions are", "significantly different (p<0.05)" if p_kruskal < 0.05
      else "NOT significantly different")
 
# pairwise mann-whitney U (bonferroni corrected with 6 pairs)
print("\npairwise mann-whitney U (bonferroni α=0.05/6=0.0083):")
pairs = [(reg_order[i], reg_order[j])
         for i in range(len(reg_order)) for j in range(i+1, len(reg_order))]
for r1, r2 in pairs:
    a = df[df["region"] == r1]["vader_compound"].values
    b = df[df["region"] == r2]["vader_compound"].values
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    sig = "**" if p < 0.0083 else ("*" if p < 0.05 else "ns")
    print(f"  {r1:<25} vs {r2:<25}  p={p:.4f} {sig}")
 
# violin plot: distribution overview
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(
    data=df, x="region", y="vader_compound",
    order=reg_order, palette=palette,
    inner=None, cut=0, linewidth=0.8, ax=ax, alpha=0.7
)
for i, region in enumerate(reg_order):
    sub = df[df["region"] == region]["vader_compound"]
    jitter = np.random.uniform(-0.2, 0.2, size=len(sub))
    ax.scatter(i + jitter, sub, color=reg_colors[region],
               s=1.5**2, alpha=0.25, zorder=1)
    
# add mean markers
for i, r in enumerate(reg_order):
    m = df[df["region"] == r]["vader_compound"].mean()
    ax.scatter(i, m, color="white", s=60, zorder=5, edgecolors="black", linewidths=0.8)
 
ax.set_facecolor("linen")
ax.axhline(0.05,  color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="±0.05 threshold")
ax.axhline(-0.05, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(0,     color="black", linewidth=0.5, alpha=0.4)
ax.set_xlabel("")
ax.set_ylabel("vader compound scores")
#ax.set_title("sentiment score distribution per region")
ax.set_xticklabels(reg_order, rotation=15, ha="right")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("vader_violin.png", dpi=180, bbox_inches="tight")
plt.close()
 
# stacked bar: label distribution per region
label_pct_plot = (
    df.groupby("region")["vader_label"]
    .value_counts(normalize=True)
    .mul(100)
    .unstack(fill_value=0)
    .reindex(reg_order)
    [["positive", "neutral", "negative"]]
)
fig, ax = plt.subplots(figsize=(8, 5))
label_pct_plot.plot(
    kind="bar", stacked=True, ax=ax,
    color=["#4CAF50", "#9E9E9E", "#F44336"],
    edgecolor="white", linewidth=0.5
)
ax.set_facecolor("linen")
ax.set_xlabel("")
ax.set_ylabel("% articles")
ax.set_title("sentiment label distribution per region") # aka vader
ax.set_xticklabels(reg_order, rotation=15, ha="right")
ax.legend(title="sentiment", bbox_to_anchor=(1, 1), loc="upper left")
ax.set_ylim(0, 105)
 
# annotation of each bar segment
for i, region in enumerate(reg_order):
    cumsum = 0
    for label, color in [("positive", "mediumseagreen"), ("neutral", "lightslategray"), ("negative", "indianred")]:
        val = label_pct_plot.loc[region, label]
        if val > 5:
            ax.text(i, cumsum + val / 2, f"{val:.0f}%",
                    ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        cumsum += val
 
plt.tight_layout()
plt.savefig("vader_labeldistribution.png", dpi=180, bbox_inches="tight")
plt.close()
 
# compound scores
subscore_means = (
    df.groupby("region")[["vader_pos", "vader_neu", "vader_neg"]]
    .mean()
    .reindex(reg_order)
)
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(reg_order))
w = 0.25
ax.bar(x - w, subscore_means["vader_pos"], width=w, color="mediumseagreen", label="positive")
ax.bar(x, subscore_means["vader_neu"], width=w, color="lightslategray", label="neutral")
ax.bar(x + w, subscore_means["vader_neg"], width=w, color="indianred", label="negative")
ax.set_facecolor("linen")
ax.set_xticks(x)
ax.set_xticklabels(reg_order, rotation=15, ha="right")
ax.set_ylabel("mean proportion of text")
#ax.set_title("vader subscores per region\n(pos/neu/neg as proportion of words)")
ax.legend()
plt.tight_layout()
plt.savefig("vader_subscores.png", dpi=180, bbox_inches="tight")
plt.close()

# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html

In [ ]:
# ROBERTA modeling
# https://huggingface.co/siebert/sentiment-roberta-large-english

from transformers import pipeline
import torch
from tqdm import tqdm

# CPU with MPS acceleration
device = 0 if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else -1
)

classifier = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english", # use siebert model
    device=device,
    truncation=True,
    max_length=512
)

def chunk_and_score(text, max_len=512):
    words = text.split() # ROBERTA has max of 512 token chunks
    chunks = [" ".join(words[i:i+400]) for i in range(0, len(words), 400)] # split into 400 token chunks and return average score
    scores = []
    for chunk in chunks:
        result = classifier(chunk)[0]
        # convert to -1/+1 scale (to compare with VADER)
        score = result["score"] if result["label"] == "POSITIVE" else -result["score"]
        scores.append(score)
    return sum(scores) / len(scores)

df = pd.read_csv("MASTER_SAMPLED.csv")
tqdm.pandas()
df["roberta_score"] = df["full_text"].progress_apply(
    lambda x: chunk_and_score(str(x))
)
df["roberta_label"] = df["roberta_score"].apply(
    lambda x: "positive" if x > 0.05 else ("negative" if x < -0.05 else "neutral")
)

df.to_csv("robertaresults.csv", index=False)
print(df.groupby("region")[["roberta_score"]].agg(["mean", "std", "count"]))

In [ ]:
# PCA plots (same as VADER) and bar plot
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv("robertaresults.csv")
df["date_parsed"] = pd.to_datetime(df["date"], format="%a, %d %b %Y %H:%M:%S %Z", utc=True)
df = df[df["date_parsed"] >= "2025-01-01"].copy()
df = df.sort_values("date_parsed").reset_index(drop=True)

reg_order  = ["US", "affected europe", "greenland & denmark", "other"]
reg_colors = {
    "US":                   "indianred",
    "affected europe":      "darkgoldenrod",
    "greenland & denmark":  "darkseagreen",
    "other":                "lightslategrey",
}
palette = [reg_colors[r] for r in reg_order]

# violin + strip plot (distribution overview) 
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(
    data=df, x="region", y="roberta_score",
    order=reg_order, palette=palette,
    inner=None, cut=0, linewidth=0.8, ax=ax, alpha=0.7
)
for i, region in enumerate(reg_order):
    sub = df[df["region"] == region]["roberta_score"]
    jitter = np.random.uniform(-0.2, 0.2, size=len(sub))
    ax.scatter(i + jitter, sub, color=reg_colors[region],
               s=1.5**2, alpha=0.25, zorder=1)
    
# add mean markers
for i, r in enumerate(reg_order):
    m = df[df["region"] == r]["roberta_score"].mean()
    ax.scatter(i, m, color="white", s=60, zorder=5, edgecolors="black", linewidths=0.8)
 
ax.set_facecolor("linen")
ax.axhline(0.05,  color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="±0.05 threshold")
ax.axhline(-0.05, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(0,     color="black", linewidth=0.5, alpha=0.4)
ax.set_xlabel("")
ax.set_ylabel("roberta scores")
#ax.set_title("sentiment score distribution per region") # white dot = mean
ax.set_xticklabels(reg_order, rotation=15, ha="right")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("roberta_violin.png", dpi=180, bbox_inches="tight")
plt.close()

# stacked bar — label distribution per region
label_pct_plot = (
    df.groupby("region")["roberta_label"]
    .value_counts(normalize=True)
    .mul(100)
    .unstack(fill_value=0)
    .reindex(reg_order)
    [["positive", "neutral", "negative"]]   # explicit order
)
fig, ax = plt.subplots(figsize=(8, 5))
label_pct_plot.plot(
    kind="bar", stacked=True, ax=ax,
    color=["#4CAF50", "#9E9E9E", "#F44336"],
    edgecolor="white", linewidth=0.5
)
ax.set_facecolor("linen")
ax.set_xlabel("")
ax.set_ylabel("% articles")
ax.set_title("sentiment label distribution per region") # aka roberta
ax.set_xticklabels(reg_order, rotation=15, ha="right")
ax.legend(title="sentiment", bbox_to_anchor=(1, 1), loc="upper left")
ax.set_ylim(0, 105)
 
# annotation of each bar segment
for i, region in enumerate(reg_order):
    cumsum = 0
    for label, color in [("positive", "mediumseagreen"), ("neutral", "lightslategray"), ("negative", "indianred")]:
        val = label_pct_plot.loc[region, label]
        if val > 5:
            ax.text(i, cumsum + val / 2, f"{val:.0f}%",
                    ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        cumsum += val
 
plt.tight_layout()
plt.savefig("roberta_labeldistribution.png", dpi=180, bbox_inches="tight")
plt.close()

# temporal
df["week_dt"] = df["date_parsed"].dt.to_period("W").dt.start_time
weekly = (
    df.groupby(["week_dt", "region"])["roberta_score"]
    .agg(mean="mean", count="count")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
for region in reg_order:
    sub = weekly[weekly["region"] == region].sort_values("week_dt")
    sub = sub[sub["count"] >= 2]
    ax.plot(sub["week_dt"], sub["mean"],
            color=reg_colors[region], linewidth=1.2, alpha=0.4, linestyle="--")
    if len(sub) >= 4:
        smoothed = sub["mean"].rolling(3, center=True, min_periods=2).mean()
        ax.plot(sub["week_dt"], smoothed,
                color=reg_colors[region], linewidth=2.5, label=region)
ax.set_facecolor("linen")
ax.axhline(0.05,  color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
ax.axhline(-0.05, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
ax.axhline(0,     color="black", linewidth=0.6, alpha=0.4)

events = {
    "Trump first\nGreenland claim": "2025-01-07",
}
for label, date in events.items():
    ts = pd.Timestamp(date, tz="UTC")
    ax.axvline(ts, color="black", linewidth=1, linestyle="-.", alpha=0.55)
    ax.text(ts, 1.02, label, fontsize=7, rotation=90,
            va="bottom", ha="right", transform=ax.get_xaxis_transform(), alpha=0.7)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")
ax.set_ylabel("mean RoBERTa score")
ax.set_title("weekly sentiment per region\ndashed: raw weekly mean, solid: 3-week rolling avg")
ax.legend(fontsize=9, loc="lower left")
ax.set_ylim(-1.05, 1.05)
plt.tight_layout()
plt.savefig("roberta_temporal.png", dpi=180, bbox_inches="tight")
plt.close()

In [ ]:
# VADER vs ROBERTA means bar plot and correlation
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html

plt.style.use('seaborn-v0_8-whitegrid')

try:
    vader_df = pd.read_csv("vadersentiment.csv")[["title", "date", "vader_compound"]]
    merged = df.merge(vader_df, on=["title", "date"], how="inner")

    fig, ax = plt.subplots(figsize=(7, 5))
    r, pv = stats.pearsonr(merged["vader_compound"], merged["roberta_score"])

    vader_means   = merged.groupby("region")["vader_compound"].mean().reindex(reg_order)
    roberta_means = merged.groupby("region")["roberta_score"].mean().reindex(reg_order)
    x = np.arange(len(reg_order))
    w = 0.35
    
    ax.bar(x - w/2, vader_means,   width=w, color="lightslategrey", label="VADER",   alpha=0.85)
    ax.bar(x + w/2, roberta_means, width=w, color="darkgoldenrod",  label="RoBERTa", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(reg_order, rotation=15, ha="right")
    ax.set_ylabel("mean sentiment score")
    ax.set_facecolor("linen")
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.legend()
    ax.set_title("mean sentiment: VADER vs RoBERTa")

    plt.tight_layout()
    plt.savefig("vader_vs_roberta.png", dpi=180, bbox_inches="tight")
    plt.close()
    
    print(f"VADER & RoBERTa Pearson r={r:.3f}")
    print("\nper-region correlation:")
    for region in reg_order:
        sub = merged[merged["region"] == region]
        rc, rpv = stats.pearsonr(sub["vader_compound"], sub["roberta_score"])
        print(f"  {region:<25}  r={rc:.3f}  p={'<0.001' if rpv<0.001 else f'{rpv:.3f}'}")
except FileNotFoundError:
    print("vadersentiment.csv not found — skipping comparison plot")

In [ ]:
# stats for ROBERTA (kruskal-wallis , mann-whitney U)
# same references as for VADER

df = pd.read_csv("robertaresults.csv")
reg_order = ["US", "affected europe", "greenland & denmark", "other"]

groups = [
    df[df["region"] == r]["roberta_score"].values
    for r in reg_order
]

summary = (
    df.groupby("region")["roberta_score"]
    .agg(["mean", "std", "count"])
    .reindex(reg_order)
    .rename(columns={"mean": "avg_compound", "std": "sd", "count": "n"})
    .round(4)
)
label_pct = (
    df.groupby("region")["roberta_label"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .unstack(fill_value=0)
    .reindex(reg_order)
)

summary = summary.join(label_pct)
print("\nSUMMARY TABLE:")
print(summary.to_string())

H_stat, p_kruskal = stats.kruskal(*groups)
print(f"\nkruskal-wallis H={H_stat:.3f}, p={p_kruskal:.4f}")
print("regions are", "significantly different ( p<0.05 )" if p_kruskal < 0.05
      else "NOT significantly different")

print("\npairwise mann-whitney U (bonferroni α=0.05/6=0.0083):")
pairs = [(reg_order[i], reg_order[j])
         for i in range(len(reg_order)) for j in range(i+1, len(reg_order))]
for r1, r2 in pairs:
    a = df[df["region"] == r1]["roberta_score"].values
    b = df[df["region"] == r2]["roberta_score"].values
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    sig = "**" if p < 0.0083 else ("*" if p < 0.05 else "ns")
    print(f"  {r1:<25} vs {r2:<25}  p={p:.4f} {sig}")

In [ ]:
# roberta example excerpts

df = pd.read_csv("robertaresults.csv")
reg_order = ["US", "affected europe", "greenland & denmark", "other"]

EXCERPT_LEN = 300  # appropriate length of excerpt

def clean_excerpt(text, n=EXCERPT_LEN):
    text = str(text).strip().replace("\n", " ") # clean first n characters ending at a word boundary
    if len(text) <= n:
        return text
    return text[:n].rsplit(" ", 1)[0] + "..."

for region in reg_order:
    sub = df[df["region"] == region].copy()
    sub["ex_score"] = sub["roberta_score"].abs()

    # example per category
    pos = sub[sub["roberta_label"] == "positive"].nlargest(1, "roberta_score").iloc[0]
    neg = sub[sub["roberta_label"] == "negative"].nsmallest(1, "roberta_score").iloc[0]
    neu = sub[sub["roberta_label"] == "neutral"].nsmallest(1, "ex_score").iloc[0]

    print(f"region: {region.upper()}")

    print(f"\n[positive score: {pos['roberta_score']:.3f}]")
    print(f"outlet: {pos['domain']}")
    print(f"title: {pos['title']}")
    print(f"excerpt: {clean_excerpt(pos['full_text'])}")

    print(f"\n[neutral score: {neu['roberta_score']:.4f}]")
    print(f"outlet: {neu['domain']}")
    print(f"title: {neu['title']}")
    print(f"excerpt: {clean_excerpt(neu['full_text'])}")

    print(f"\n[negative score: {neg['roberta_score']:.3f}]")
    print(f"outlet: {neg['domain']}")
    print(f"title: {neg['title']}")
    print(f"excerpt: {clean_excerpt(neg['full_text'])}")